# Environment change

Este notebook migra el modelo y sus datos de línea base desde el esquema de desarrollo
hacia el esquema de producción. Es el paso intermedio entre la creación de baselines (16)
y la inferencia en producción (18). Copia la versión PRODUCTION del modelo, replica las
tablas de histogramas y métricas base, y aplica un tag CANDIDATE_VERSION al modelo
en el esquema destino para que el notebook de inferencia sepa qué versión ejecutar.

## 1. Setup

Configuración inicial: sesión de Snowpark, constantes de esquemas origen/destino,
nombres de tablas de línea base y columnas de agregación.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.ml.registry import Registry
session = get_active_session()

### 1A. Constants

Constantes que definen los esquemas origen (desarrollo) y destino (producción),
las tablas de línea base a sincronizar, y el tag CANDIDATE_VERSION que marca
la versión del modelo lista para inferencia.

In [ ]:
SRC_DATABASE = "BD_AA_DEV"
TGT_DATABASE = "BD_AA_DEV"

# Source schemas (development)
SRC_STORAGE_SCHEMA = "SC_STORAGE_BMX_PS"
SRC_MODELS_SCHEMA = "SC_MODELS_BMX"

# Target schemas (production)
TGT_STORAGE_SCHEMA = "SC_FEATURES_BMX"
TGT_MODELS_SCHEMA = "SC_STORAGE_BMX_PS"

# Model
MODEL_NAME = "UNI_BOX_REGRESSION_PARTITIONED"
SRC_MODEL_FQN = f"{SRC_DATABASE}.{SRC_MODELS_SCHEMA}.{MODEL_NAME}"
TGT_MODEL_FQN = f"{TGT_DATABASE}.{TGT_MODELS_SCHEMA}.{MODEL_NAME}"

# Baseline tables (source)
SRC_DATA_DRIFT_BASELINE = f"{SRC_DATABASE}.{SRC_STORAGE_SCHEMA}.DA_DATA_DRIFT_HISTOGRAMS_BASELINE"
SRC_PRED_DRIFT_BASELINE = f"{SRC_DATABASE}.{SRC_STORAGE_SCHEMA}.DA_PREDICTION_DRIFT_HISTOGRAMS_BASELINE"
SRC_PERF_BASELINE = f"{SRC_DATABASE}.{SRC_STORAGE_SCHEMA}.DA_PERFORMANCE_BASELINE"

# Baseline tables (target)
TGT_DATA_DRIFT_BASELINE = f"{TGT_DATABASE}.{TGT_STORAGE_SCHEMA}.DA_DATA_DRIFT_HISTOGRAMS_BASELINE"
TGT_PRED_DRIFT_BASELINE = f"{TGT_DATABASE}.{TGT_STORAGE_SCHEMA}.DA_PREDICTION_DRIFT_HISTOGRAMS_BASELINE"
TGT_PERF_BASELINE = f"{TGT_DATABASE}.{TGT_STORAGE_SCHEMA}.DA_PERFORMANCE_BASELINE"

# Tag for candidate model versions
CANDIDATE_TAG_FQN = f"{TGT_DATABASE}.{TGT_MODELS_SCHEMA}.CANDIDATE_VERSION"

# Aggregation columns used in histogram/perf tables
AGG_COLS = ["STATS_NTILE_GROUP", "CUST_CATEGORY"]

# Performance metric names (used to check missing combos in perf table)
PERF_METRIC_NAMES = ["wape", "rmse", "mae", "f1_binary"]

session.sql(f"USE DATABASE {SRC_DATABASE}").collect()

## 2. Migrate PRODUCTION model version

Resuelve el alias PRODUCTION a una versión concreta en el esquema de desarrollo y la
copia al esquema de producción usando CREATE MODEL / ALTER MODEL ADD VERSION.
Si la versión ya existe en el destino, se salta la migración.
Después aplica el tag CANDIDATE_VERSION al modelo para señalar cuál versión debe
usarse en la inferencia (notebook 18).

In [ ]:
# Utiliza el alias PARTICION para identificar la version del modelo que es relevante
src_registry = Registry(
    session=session,
    database_name=SRC_DATABASE,
    schema_name=SRC_MODELS_SCHEMA,
)
src_model_ref = src_registry.get_model(MODEL_NAME)
prod_version = src_model_ref.version("PRODUCTION")
prod_version_name = prod_version.version_name

print(f"Source PRODUCTION alias -> version: {prod_version_name}")

In [ ]:
# Revisa si el modelo ya existe en el otro ambiente. Si no, copia la versión
# PRODUCTION; de otra manera agrega la version PRODUCTION al modelo

tgt_registry = Registry(
    session=session,
    database_name=TGT_DATABASE,
    schema_name=TGT_MODELS_SCHEMA,
)

try:
    tgt_model_ref = tgt_registry.get_model(MODEL_NAME)
    existing_versions = {v.version_name for v in tgt_model_ref.versions()}
    model_exists = True
except Exception:
    existing_versions = set()
    model_exists = False

if prod_version_name in existing_versions:
    print(f"Version {prod_version_name} already exists in {TGT_MODEL_FQN}, skipping migration.")
else:
    if model_exists:
        # Model object exists but this version is missing — add the version
        add_version_sql = f"""
        ALTER MODEL {TGT_MODEL_FQN} ADD VERSION {prod_version_name}
            FROM MODEL {SRC_MODEL_FQN} VERSION {prod_version_name}
        """
        session.sql(add_version_sql).collect()
        print(f"Added version {prod_version_name} to existing model {TGT_MODEL_FQN}")
    else:
        # Model does not exist — create it with this version
        create_model_sql = f"""
        CREATE MODEL {TGT_MODEL_FQN} WITH VERSION {prod_version_name}
            FROM MODEL {SRC_MODEL_FQN} VERSION {prod_version_name}
        """
        session.sql(create_model_sql).collect()
        print(f"Created model {TGT_MODEL_FQN} with version {prod_version_name}")

In [ ]:
# Create the tag if it doesn't exist and apply it to the model
session.sql(f"CREATE TAG IF NOT EXISTS {CANDIDATE_TAG_FQN}").collect()

session.sql(f"""
    ALTER MODEL {TGT_MODEL_FQN}
        SET TAG {CANDIDATE_TAG_FQN} = '{prod_version_name}'
""").collect()

print(f"Applied tag {CANDIDATE_TAG_FQN} = '{prod_version_name}' on {TGT_MODEL_FQN}")

## 3. Create baseline tables if missing

Crea las tablas de línea base en el esquema de producción (si no existen)
usando la estructura de las tablas origen con CREATE TABLE ... LIKE.

In [ ]:
# For each baseline table, create the target table LIKE the source if it doesn't exist.

baseline_pairs = [
    (SRC_DATA_DRIFT_BASELINE, TGT_DATA_DRIFT_BASELINE),
    (SRC_PRED_DRIFT_BASELINE, TGT_PRED_DRIFT_BASELINE),
    (SRC_PERF_BASELINE, TGT_PERF_BASELINE),
]

for src_tbl, tgt_tbl in baseline_pairs:
    session.sql(f"CREATE TABLE IF NOT EXISTS {tgt_tbl} LIKE {src_tbl}").collect()
    print(f"Table ready: {tgt_tbl}")

## 4. Insert missing baseline data

Para cada tabla de línea base, identifica combinaciones (MODEL_NAME, MODEL_VERSION,
AGGREGATED_COL) que existen en origen pero faltan en destino, e inserta las filas
correspondientes. Esto asegura que el esquema de producción tenga todos los datos
de referencia necesarios para el monitoreo (notebook 19).

In [ ]:

sync_pairs = [
    (SRC_DATA_DRIFT_BASELINE, TGT_DATA_DRIFT_BASELINE),
    (SRC_PRED_DRIFT_BASELINE, TGT_PRED_DRIFT_BASELINE),
    (SRC_PERF_BASELINE, TGT_PERF_BASELINE),
]

for src_tbl, tgt_tbl in sync_pairs:
    src_combos = (
        session.table(src_tbl)
        .filter(F.col("MODEL_NAME") == MODEL_NAME)
        .select("MODEL_NAME", "MODEL_VERSION", "AGGREGATED_COL")
        .distinct()
    )

    tgt_combos = (
        session.table(tgt_tbl)
        .filter(F.col("MODEL_NAME") == MODEL_NAME)
        .select("MODEL_NAME", "MODEL_VERSION", "AGGREGATED_COL")
        .distinct()
    )

    missing_combos = src_combos.join(
        tgt_combos,
        on=["MODEL_NAME", "MODEL_VERSION", "AGGREGATED_COL"],
        how="left_anti",
    )

    missing_list = missing_combos.collect()

    if not missing_list:
        print(f"{tgt_tbl}: no missing combos, skipping.")
        continue

    for row in missing_list:
        mv = row["MODEL_VERSION"]
        ac = row["AGGREGATED_COL"]

        rows_to_insert = (
            session.table(src_tbl)
            .filter(F.col("MODEL_NAME") == MODEL_NAME)
            .filter(F.col("MODEL_VERSION") == mv)
            .filter(F.col("AGGREGATED_COL") == ac)
        )

        rows_to_insert.write.mode("append").save_as_table(tgt_tbl)
        count = rows_to_insert.count()
        print(f"  Inserted {count:,} rows into {tgt_tbl} for version={mv}, agg_col={ac}")

    total = session.table(tgt_tbl).count()
    print(f"{tgt_tbl} now has {total:,} rows.")

## Done

La versión del modelo y los datos de línea base han sido migrados a los esquemas de producción.
El siguiente paso es ejecutar la inferencia sobre datos de producción (notebook 18).